# Eval Trained Adapter v13 — Diagnostic on Stratified Holdout (LB=0.84)

Same stratified 5-per-category eval as dgxchen, but on `gideonnazarian/v13-dgxchen-boxedloss-adapter` (v13 = dgxchen recipe + BOXED_LOSS_WEIGHT=5.0, LB 0.84 — flat vs dgxchen direct 0.84).

**The question this eval answers:** v13 scored IDENTICALLY to dgxchen in aggregate. Two possibilities:
- (a) **BOXED_LOSS_WEIGHT inert** — model trained identically to dgxchen, predictions are the same.
- (b) **BOXED_LOSS_WEIGHT shifted behavior but wins/losses cancelled** — per-category accuracies differ from dgxchen, even though aggregate is the same.

If (a): drop loss-weighting entirely for v14, try a data-side or hyperparameter delta.
If (b): the lever is shifting things just not in our favor — try a smaller weight (3.0) or different gate token (`</think>` instead of `\boxed{`).

**Comparison target:** dgxchen eval (LB 0.84) — `kaggle/eval-trained/12-eval-trained-adapter-dgxchen.ipynb`. Same holdout, same scoring, same prompt. Side-by-side per-category breakdown will tell us which interpretation is right.

**Key fix vs v12 eval:** scoring uses the competition grader's logic — `math.isclose(rel_tol=1e-2, abs_tol=1e-5)` for float categories (gravity, unit_conversion), exact match for the rest. v12 eval had a measurement bug (used strict string match for floats), reporting gravity 0/5 when the model actually got 5/5 by grader logic. v13+ evals use this corrected scoring.

**The v13 adapter is packaged as `submission.zip`** (not raw adapter files like dgxchen's). Cell 2 unzips it before PEFT loading.

**Reference:**
- v9 (LB 0.62): 3/5 strict, 4/5 substring (5-problem)
- v12 (Kh0a recipe, LB 0.72): 73.3% grader-correct (30-problem)
- dgxchen (LB 0.84): see eval `kaggle/eval-trained/12-eval-trained-adapter-dgxchen.ipynb`
- **v13 (LB 0.84): this run** (30-problem stratified, grader-correct scoring)

In [ ]:
# Cell 1: Setup (sitedir + ptxas) + FAIL-FAST GPU CHECK
#
# Kaggle's machine_shape metadata is best-effort. If we get dispatched to a
# Tesla P100 16 GB (the free-tier fallback), the 30B BF16 model (~63 GB) cannot
# fit and the loader hits a MoE disk-offload re-save error after ~5 min of
# wasted work. Fail fast at the top so we know to manually fix the GPU.

import os, random, shutil, site, stat, sys, time, glob, zipfile
import numpy as np
import torch

# --- GPU check ---
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA device. Set Accelerator → GPU in the Kaggle UI sidebar.")

gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_vram_gb:.1f} GB)")

if gpu_vram_gb < 80:
    raise RuntimeError(
        f"Wrong GPU dispatched: {gpu_name} ({gpu_vram_gb:.1f} GB). "
        f"Need RTX Pro 6000 (96 GB) for 30B BF16. "
        f"FIX: open this kernel in the Kaggle UI, click 'Notebook options' or 'Settings' in the right sidebar, "
        f"change Accelerator to 'RTX Pro 6000', then click Save & Run All again. "
        f"The machine_shape in kernel-metadata.json is best-effort; Kaggle sometimes falls back to P100/T4."
    )

T0 = time.perf_counter()
def step(label):
    print(f"[{time.perf_counter() - T0:7.1f}s] {label}", flush=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

for utility_root in ("/kaggle/input/notebooks", "/kaggle/usr/lib/notebooks", "/kaggle/lib"):
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _ in os.walk(utility_root):
        if "models" in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == "python_packages":
            site.addsitedir(root)

PTXAS_DST = "/tmp/ptxas-blackwell"
for src in (
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
):
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(PTXAS_DST, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_DST
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = PTXAS_DST
        break
step("Setup done")

In [ ]:
# Cell 2: Locate v13 adapter (unzipping submission.zip if needed), then load base model + attach via PeftModel
import kagglehub
import mamba_ssm  # noqa: F401
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# v13 adapter dataset paths to probe. The dataset contains submission.zip which contains v13_adapter/.
V13_DATASET_CANDIDATES = [
    "/kaggle/input/datasets/gideonnazarian/v13-dgxchen-boxedloss-adapter",
    "/kaggle/input/v13-dgxchen-boxedloss-adapter",
]
V13_DATASET = None
for p in V13_DATASET_CANDIDATES:
    if os.path.isdir(p):
        V13_DATASET = p
        break
if V13_DATASET is None:
    raise FileNotFoundError(
        f"v13 adapter dataset not found in any expected path. /kaggle/input/ has: {os.listdir('/kaggle/input')}"
    )
step(f"V13_DATASET={V13_DATASET}")
print("  Contents:", os.listdir(V13_DATASET))

# Locate adapter — either raw adapter files OR submission.zip that needs unzipping
ADAPTER_PATH = None

# Path 1: raw adapter files at the dataset root (unlikely for v13 but possible)
if os.path.exists(os.path.join(V13_DATASET, "adapter_config.json")):
    ADAPTER_PATH = V13_DATASET
    step(f"adapter_config.json found directly at dataset root")

# Path 2: submission.zip → unzip to /kaggle/working and find adapter inside
if ADAPTER_PATH is None:
    zip_path = os.path.join(V13_DATASET, "submission.zip")
    if os.path.exists(zip_path):
        extract_dir = "/kaggle/working/v13_extracted"
        os.makedirs(extract_dir, exist_ok=True)
        step(f"Extracting {zip_path} → {extract_dir}")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)
        # Find adapter_config.json inside extracted tree
        hits = glob.glob(os.path.join(extract_dir, "**", "adapter_config.json"), recursive=True)
        if hits:
            ADAPTER_PATH = os.path.dirname(hits[0])
            step(f"Found adapter inside zip at {ADAPTER_PATH}")

# Path 3 (fallback): glob for adapter_config.json anywhere under /kaggle/input/
if ADAPTER_PATH is None:
    hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if hits:
        ADAPTER_PATH = os.path.dirname(hits[0])
        step(f"Fallback glob found adapter at {ADAPTER_PATH}")

if ADAPTER_PATH is None:
    raise FileNotFoundError(
        f"v13 adapter_config.json not found. Inspect dataset structure: {os.listdir(V13_DATASET)}"
    )

print("--- v13 adapter contents ---")
for f in sorted(os.listdir(ADAPTER_PATH)):
    full = os.path.join(ADAPTER_PATH, f)
    if os.path.isfile(full):
        print(f"  {f}: {os.path.getsize(full):>12,} bytes")

# Confirm adapter config matches v13 (r=32, alpha=32, 8 modules)
import json as _json
with open(os.path.join(ADAPTER_PATH, "adapter_config.json")) as f:
    cfg = _json.load(f)
print(f"\nadapter config sanity: r={cfg.get('r')} alpha={cfg.get('lora_alpha')} "
      f"dropout={cfg.get('lora_dropout')} targets={cfg.get('target_modules')}")

# Load base model + tokenizer
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
step(f"Tokenizer loaded; IM_END_ID={IM_END_ID}")

t = time.perf_counter()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16,
)
step(f"Base model loaded in {time.perf_counter() - t:.0f}s")

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
step(f"v13 adapter attached via PeftModel.from_pretrained")
step(f"VRAM={torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# Cell 3: Stratified per-category eval (5 per category × 6 categories = 30 problems)
# Identical sampling to dgxchen eval (seed=42) for direct row-by-row comparability.
import re
import json
import pandas as pd
import random

random.seed(42)

PROMPT_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)
SAMPLES_PER_CATEGORY = 5
MAX_NEW_TOKENS = 4096   # generous; v13 max_seq=8192 so generation up to ~4k is safe

TRAIN_CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(TRAIN_CSV)

def classify(prompt: str) -> str:
    p = prompt.lower()
    if "bit manipulation" in p: return "bit_manipulation"
    if "secret encryption rules" in p: return "cipher"
    if "unit conversion" in p: return "unit_conversion"
    if "numeral system" in p: return "numeral_system"
    if "gravitational constant" in p: return "gravity"
    if "transformation rules" in p: return "transformation_rules"
    return "unknown"

df["category"] = df["prompt"].apply(classify)
print("Category counts:", df["category"].value_counts().to_dict())

holdout_ids = []
for cat in sorted(df["category"].unique()):
    if cat == "unknown": continue
    cat_df = df[df["category"] == cat]
    n = min(SAMPLES_PER_CATEGORY, len(cat_df))
    sampled = cat_df.sample(n=n, random_state=42)
    holdout_ids.extend(sampled["id"].tolist())
holdout = df[df["id"].isin(holdout_ids)].copy()
step(f"Stratified holdout: {len(holdout)} problems across {holdout['category'].nunique()} categories")
for cat in sorted(holdout["category"].unique()):
    print(f"  {cat}: {len(holdout[holdout['category']==cat])}")

BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")
THINK_CLOSE = "</think>"
IM_END = "<|im_end|>"

def extract_after_think(c):
    if THINK_CLOSE not in c: return ""
    post = c.split(THINK_CLOSE, 1)[1]
    if IM_END in post: post = post.split(IM_END, 1)[0]
    return post.strip()

def extract_boxed_anywhere(c):
    matches = list(BOXED_RE.finditer(c))
    return matches[-1].group(1).strip() if matches else ""

def gold_substring(c, g):
    return g if (g and g in c) else ""

results = []
for i, row in holdout.iterrows():
    pid = row["id"]; gold = str(row["answer"]); cat = row["category"]
    prompt_text = row["prompt"] + PROMPT_SUFFIX
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        return_tensors="pt", add_generation_prompt=True, return_dict=True,
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    t_p = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            eos_token_id=[IM_END_ID, tokenizer.eos_token_id],
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - t_p
    n_new = out.shape[1] - prompt_len
    completion = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=False)
    after = extract_after_think(completion)
    boxed = extract_boxed_anywhere(completion)
    substr = gold_substring(completion, gold)
    results.append({
        "id": pid, "category": cat, "completion_tokens": n_new, "seconds": elapsed,
        "after_think": after, "boxed_anywhere": boxed, "gold": gold,
        "completion": completion,
        "correct_after_think": after.strip() == gold.strip(),
        "correct_boxed": boxed.strip() == gold.strip(),
        "correct_substring": bool(substr),
        "has_think_close": THINK_CLOSE in completion,
    })
    mark = "✓" if boxed.strip()==gold.strip() else ("~" if substr else "✗")
    print(
        f"  [{len(results):2d}/{len(holdout)}] {cat:22s} {pid} | {n_new:4d} tok in {elapsed:5.1f}s | "
        f"boxed={boxed!r:25s} gold={gold!r:15s} {mark}",
        flush=True,
    )

In [ ]:
# Cell 4: Per-category summary using GRADER'S ACTUAL LOGIC (isclose for floats)
import json
import math
import pandas as pd
from collections import defaultdict

FLOAT_CATS = {"gravity", "unit_conversion"}

def is_grader_correct(boxed, gold, category):
    if not boxed or not isinstance(boxed, str): return False
    b = boxed.strip(); g = str(gold).strip()
    if category in FLOAT_CATS:
        try:
            return math.isclose(float(b), float(g), rel_tol=1e-2, abs_tol=1e-5)
        except (ValueError, TypeError):
            return b.lower() == g.lower()
    return b.lower() == g.lower()

n = len(results)
print("\n=== v13 Adapter (dgxchen recipe + BOXED_LOSS_WEIGHT=5.0, LB 0.84) — Stratified Eval ===")
print(f"Total: {n} problems\n")

per_cat = defaultdict(lambda: {"n": 0, "strict_old": 0, "grader_correct": 0, "substring": 0, "close": 0})
for r in results:
    c = r["category"]
    per_cat[c]["n"] += 1
    per_cat[c]["strict_old"] += int(r["correct_boxed"])
    per_cat[c]["grader_correct"] += int(is_grader_correct(r["boxed_anywhere"], r["gold"], c))
    per_cat[c]["substring"] += int(r["correct_substring"])
    per_cat[c]["close"] += int(r["has_think_close"])

print(f"{'category':22s} | strict-old | grader-correct | substring | think_close")
print("-" * 90)
for cat in sorted(per_cat):
    s = per_cat[cat]
    print(
        f"{cat:22s} | {s['strict_old']}/{s['n']}        | "
        f"{s['grader_correct']}/{s['n']}            | "
        f"{s['substring']}/{s['n']}      | "
        f"{s['close']}/{s['n']}"
    )

agg_strict_old = sum(s["strict_old"] for s in per_cat.values()) / n
agg_grader = sum(s["grader_correct"] for s in per_cat.values()) / n
agg_substr = sum(s["substring"] for s in per_cat.values()) / n
print(f"\nAggregate: strict-old={agg_strict_old:.1%}, grader-correct={agg_grader:.1%}, substring={agg_substr:.1%}")

print("\n=== Reference for v14 decision tree ===")
print(f"dgxchen (LB 0.84): per-category breakdown in kaggle/eval-trained/12-eval-trained-adapter-dgxchen.ipynb")
print(f"v13      (LB 0.84): {agg_grader:.1%} grader-correct — THIS RUN")
print("\nDecision:")
print("  - If per-category matches dgxchen → BOXED_LOSS_WEIGHT inert → v14 needs different lever")
print("  - If per-category DIFFERS but aggregate same → upweight shifted behavior → v14 = tune weight/gate")

pd.DataFrame(results).to_csv("/kaggle/working/v13_eval_results.csv", index=False)
with open("/kaggle/working/v13_eval_summary.json", "w") as f:
    json.dump({
        "n_total": n,
        "agg_strict_old": agg_strict_old,
        "agg_grader_correct": agg_grader,
        "agg_substring": agg_substr,
        "per_category": dict(per_cat),
        "note": "grader_correct applies math.isclose(rel_tol=1e-2) for gravity+unit_conversion; exact match for rest",
    }, f, indent=2)
step(f"Saved v13_eval_results.csv + v13_eval_summary.json")

In [ ]:
# Cell 5: Last 800 chars per problem for offline failure-mode analysis
for i, r in enumerate(results, 1):
    print(f"=== [{i:2d}/{len(results)}] {r['category']:22s} {r['id']} gold={r['gold']!r} ===")
    print(f"close={r['has_think_close']} | boxed={r['boxed_anywhere']!r} | substr={r['correct_substring']} | strict={r['correct_boxed']}")
    print("--- LAST 800 CHARS ---")
    print(r["completion"][-800:])
    print()
step(f"Total wall time: {(time.perf_counter() - T0) / 60:.1f} min")